Question 1

    Install uv
    What's the version of uv you installed?
    Use --version to find out



In [7]:
import uv


In [17]:
!uv --version


uv 0.9.7 (0adb44480 2025-10-30)


Question 2

    Use uv to install Scikit-Learn version 1.6.1
    What's the first hash for Scikit-Learn you get in the lock file?
    Include the entire string starting with sha256:, don't include quotes

Models

We have prepared a pipeline with a dictionary vectorizer and a model.

It was trained (roughly) using this code:

categorical = ['lead_source']
numeric = ['number_of_courses_viewed', 'annual_income']

df[categorical] = df[categorical].fillna('NA')
df[numeric] = df[numeric].fillna(0)

train_dict = df[categorical + numeric].to_dict(orient='records')

pipeline = make_pipeline(
    DictVectorizer(),
    LogisticRegression(solver='liblinear')
)

pipeline.fit(train_dict, y_train)

    Note: You don't need to train the model. This code is just for your reference.

And then saved with Pickle. Download it here.

With wget:

wget https://github.com/DataTalksClub/machine-learning-zoomcamp/raw/refs/heads/master/cohorts/2025/05-deployment/pipeline_v1.bin

In [18]:
!ls


HW5.ipynb


In [21]:
!uv init

Initialized project `hw5`


In [22]:
!uv add scikit-learn==1.6.1

Using CPython 3.12.11 interpreter at: /opt/homebrew/Caskroom/mambaforge/base/envs/dataTalks/bin/python3.12
Creating virtual environment at: .venv
Resolved 6 packages in 4ms                                           
Installed 5 packages in 20ms                                
 + joblib==1.5.2
 + numpy==2.3.4
 + scikit-learn==1.6.1
 + scipy==1.16.3
 + threadpoolctl==3.6.0


sha256:b4fc2525eca2ebfc8ccea81c6da557cab307f2c34b5f85b628e94803f9c0bf5faee

Question 3

Let's use the model!

    Write a script for loading the pipeline with pickle
    Score this record:

{
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0
}

What's the probability that this lead will convert?

    0.333
    0.533
    0.733
    0.933

If you're getting errors when unpickling the files, check their checksum:

$ md5sum pipeline_v1.bin
7d17d2e4dfbaf1e408e1a62e6e880d49 *pipeline_v1.bin

In [30]:
!md5sum pipeline_v1.bin

7d17d2e4dfbaf1e408e1a62e6e880d49  pipeline_v1.bin


In [28]:
import pickle

# Load the pipeline
with open('pipeline_v1.bin', 'rb') as f:
    pipeline = pickle.load(f)

# The record to score
record = {
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0
}

# Predict the probability
prob = pipeline.predict_proba([record])[0][1]
print(f"The probability that this lead will convert is: {prob:.3f}")

# Options: 0.333, 0.533, 0.733, 0.933
# Find the closest
options = [0.333, 0.533, 0.733, 0.933]
closest = min(options, key=lambda x: abs(x - prob))
print(f"Closest option: {closest}")


The probability that this lead will convert is: 0.534
Closest option: 0.533


Question 4

Now let's serve this model as a web service

    Install FastAPI
    Write FastAPI code for serving the model
    Now score this client using requests:

url = "YOUR_URL"
client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
}
requests.post(url, json=client).json()

What's the probability that this client will get a subscription?

    0.334
    0.534
    0.734
    0.934


In [29]:
!uv add fastapi

Resolved 17 packages in 334ms                                        
⠙ Preparing packages... (0/11)                                                  
⠙ Preparing packages... (0/11)------------------     0 B/13.32 KiB           
⠙ Preparing packages... (0/11)--------- 13.32 KiB/13.32 KiB         
⠙ Preparing packages... (0/11)--------- 13.32 KiB/13.32 KiB         
annotated-types      ------------------------------ 13.32 KiB/13.32 KiB
⠙ Preparing packages... (0/11)------------------     0 B/72.60 KiB           
annotated-types      ------------------------------ 13.32 KiB/13.32 KiB
⠙ Preparing packages... (0/11)------------------     0 B/72.60 KiB           
annotated-types      ------------------------------ 13.32 KiB/13.32 KiB
⠙ Preparing packages... (0/11)------------------     0 B/72.60 KiB           
annotated-types      ------------------------------ 13.32 KiB/13.32 KiB
starlette            ------------------------------     0 B/72.60 KiB
⠙ Preparing packages... (0/11)-----------

In [42]:
!uvicorn app:app --host 0.0.0.0 --port 8000

/opt/homebrew/Caskroom/mambaforge/base/envs/dataTalks/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/mambaforge/base/envs/dataTalks/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/mambaforge/base/envs/dataTalks/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionW

In [41]:
!python test_api.py


The probability that this client will get a subscription is: 0.534
Closest option: 0.534


Docker

Install Docker. We will use it for the next two questions.

For these questions, we prepared a base image: agrigorev/zoomcamp-model:2025. You'll need to use it (see Question 5 for an example).

This image is based on 3.13.5-slim-bookworm and has a pipeline with logistic regression (a different one) as well a dictionary vectorizer inside.

This is how the Dockerfile for this image looks like:

FROM python:3.13.5-slim-bookworm
WORKDIR /code
COPY pipeline_v2.bin .

We already built it and then pushed it to [agrigorev/zoomcamp-model:2025](https://hub.docker.com/r/agrigorev/zoomcamp-model).

    Note: You don't need to build this docker image, it's just for your reference.
